# Machine Learning Coursework 2 (6001CEM)
## Part D: Data Preparation & Feature Engineering Pipeline
**Module Code:** 6001CEM | **Credit:** 5 Credits
**Dataset:** Default of Credit Card Clients (Cleaned & Preprocessed)
---
### Preprocessing Pipeline Steps Covered:
1. **Missing Value Handling:** Systematic validation.
2. **Duplicate Removal:** Eliminating 35 duplicate client pairs.
3. **Categorical Recoding:** Standardizing `EDUCATION` (0,5,6 -> 4) and `MARRIAGE` (0 -> 3).
4. **Identifier Elimination:** Dropping non-predictive `ID` column.
5. **Domain Feature Engineering:** Credit utilization, payment-to-bill ratios, delay aggregations.
6. **Data Splitting:** Stratified 80/20 train/test split (preventing data leakage).
7. **Categorical Encoding:** One-Hot Encoding nominal demographic features.
8. **Feature Scaling:** RobustScaler applied to continuous variables.
9. **Feature Selection:** VarianceThreshold & Mutual Information ranking.
10. **PCA Dimensionality Reduction Analysis:** Evaluating variance trade-offs.
11. **Class Imbalance Treatment:** SMOTE oversampling applied strictly to training data.
12. **Dataset Export:** Saving final cleaned dataset and pipeline splits.

### 1. Library Imports & Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)
print("All pipeline libraries loaded successfully!")

df_raw = pd.read_csv('default of credit card clients.csv')
print(f"Raw dataset dimensions: {df_raw.shape}")

### 2. Missing Value & Duplicate Handling

In [ ]:
# Verify missing values
assert df_raw.isnull().sum().sum() == 0, "Missing values detected!"
print("Step 1: Missing value verification passed (0 nulls).")

# Remove duplicates (excluding ID)
feature_cols = [c for c in df_raw.columns if c != 'ID']
initial_rows = df_raw.shape[0]
df_clean = df_raw.drop_duplicates(subset=feature_cols).copy()
removed_dups = initial_rows - df_clean.shape[0]
print(f"Step 2: Dropped {removed_dups} duplicate records. Cleaned rows: {df_clean.shape[0]:,}")

### 3. Categorical Recoding & Dropping `ID`

In [ ]:
# Recode undocumented values
df_clean['EDUCATION'] = df_clean['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df_clean['MARRIAGE'] = df_clean['MARRIAGE'].replace({0: 3})

# Drop ID column
if 'ID' in df_clean.columns:
    df_clean = df_clean.drop(columns=['ID'])

print("Step 3: EDUCATION and MARRIAGE standardized.")
print("Step 4: Non-predictive 'ID' column dropped.")
print(f"Current dataframe shape: {df_clean.shape}")

### 4. Domain Feature Engineering
We construct 4 powerful domain-specific indicators to improve model predictive power:
1. **Credit Utilization Ratio (`UTIL_RATIO`):** $\frac{\text{BILL\_AMT1}}{\text{LIMIT\_BAL}}$. High credit line utilization is a primary signal of credit distress.
2. **Payment-to-Bill Ratio (`PAY_RATIO1`):** $\frac{\text{PAY\_AMT1}}{\text{BILL\_AMT2} + 1}$. Proportion of prior month's balance repaid.
3. **Max Payment Delay (`MAX_DELAY`):** $\max(\text{PAY\_1}, \dots, \text{PAY\_6})$. Captures the peak severity of default history.
4. **Total Delinquent Months (`NUM_DELAYS`):** Count of months where payment was delayed ($> 0$).

In [ ]:
# 1. Credit Utilization
df_clean['UTIL_RATIO'] = df_clean['BILL_AMT1'] / (df_clean['LIMIT_BAL'] + 1)

# 2. Payment-to-Bill Ratio (clipping negative bills)
df_clean['PAY_RATIO1'] = df_clean['PAY_AMT1'] / (df_clean['BILL_AMT2'].clip(lower=0) + 1)

# 3. Maximum Repayment Delay
pay_cols = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df_clean['MAX_DELAY'] = df_clean[pay_cols].max(axis=1)

# 4. Number of Months with Delay (> 0)
df_clean['NUM_DELAYS'] = (df_clean[pay_cols] > 0).sum(axis=1)

print("Step 5: Engineered 4 new financial risk features.")
df_clean[['UTIL_RATIO', 'PAY_RATIO1', 'MAX_DELAY', 'NUM_DELAYS']].describe().round(3)

### 5. Stratified Train-Test Split
To strictly **prevent data leakage**, all encoders, scalers, and SMOTE must be fit solely on the training partition!

In [ ]:
X = df_clean.drop(columns=['dpnm'])
y = df_clean['dpnm']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples (Default rate: {y_train.mean():.2%})")
print(f"Testing set:  {X_test.shape[0]:,} samples (Default rate: {y_test.mean():.2%})")

### 6. Categorical Encoding & Feature Scaling

In [ ]:
# Categorical nominal features
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
num_cols = [c for c in X_train.columns if c not in cat_cols]

# One-Hot Encoding
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoded_train = encoder.fit_transform(X_train[cat_cols])
encoded_test = encoder.transform(X_test[cat_cols])
encoded_feature_names = encoder.get_feature_names_out(cat_cols)

# Robust Scaling on continuous numerical features (resistant to outliers)
scaler = RobustScaler()
scaled_train = scaler.fit_transform(X_train[num_cols])
scaled_test = scaler.transform(X_test[num_cols])

# Assemble preprocessed DataFrames
X_train_prep = pd.DataFrame(
    np.hstack([scaled_train, encoded_train]),
    columns=num_cols + list(encoded_feature_names)
)
X_test_prep = pd.DataFrame(
    np.hstack([scaled_test, encoded_test]),
    columns=num_cols + list(encoded_feature_names)
)

print(f"Preprocessed feature shape: {X_train_prep.shape[1]} features.")

### 7. Feature Selection (Mutual Information)

In [ ]:
# Compute mutual information scores
mi_scores = mutual_info_classif(X_train_prep, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train_prep.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
mi_series.head(15).plot(kind='barh', color='#2b5c8f')
plt.gca().invert_yaxis()
plt.title('Top 15 Features by Mutual Information with Default', fontsize=13, fontweight='bold')
plt.xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print("Top 5 most informative features:")
print(mi_series.head(5))

### 8. PCA Dimensionality Reduction Analysis

In [ ]:
# Fit PCA to explore variance
pca = PCA()
pca.fit(X_train_prep)
cum_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', linestyle='-', color='#d95f02')
plt.axhline(y=0.90, color='gray', linestyle='--', label='90% Variance Explained')
plt.title('PCA Cumulative Explained Variance Ratio', fontsize=13, fontweight='bold')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Variance Explained')
plt.legend()
plt.tight_layout()
plt.show()

n_90 = np.argmax(cum_var >= 0.90) + 1
print(f"{n_90} components required to capture 90% of total variance.")
print("Discussion: While PCA reduces dimension, it destroys regulatory banking interpretability. Original engineered features are retained for model transparency.")

### 9. Class Imbalance Treatment: SMOTE (Synthetic Minority Over-sampling)
Applied **strictly on the training set** to balance classes without polluting the test evaluation set.

In [ ]:
print(f"Before SMOTE (Training): Class 0 = {(y_train==0).sum():,}, Class 1 = {(y_train==1).sum():,}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_prep, y_train)

print(f"After SMOTE (Training):  Class 0 = {(y_train_resampled==0).sum():,}, Class 1 = {(y_train_resampled==1).sum():,}")
print(f"Balanced training size: {X_train_resampled.shape[0]:,} samples.")

### 10. Save Cleaned & Pipeline Outputs

In [ ]:
# 1. Save standard cleaned CSV
df_clean.to_csv('default of credit card clients (cleaned).csv', index=False)
print("Exported: 'default of credit card clients (cleaned).csv'")

# 2. Save preprocessed train & test datasets
X_train_resampled['dpnm'] = y_train_resampled.values
X_test_prep['dpnm'] = y_test.values

X_train_resampled.to_csv('credit_train_preprocessed.csv', index=False)
X_test_prep.to_csv('credit_test_preprocessed.csv', index=False)

print("Exported: 'credit_train_preprocessed.csv' (SMOTE balanced)")
print("Exported: 'credit_test_preprocessed.csv' (Clean evaluation test)")
print("\n=== PREPROCESSING PIPELINE COMPLETE ===")